
# Time Series Forecasting with LSTM

A compact example of sequence-to-one regression using PyTorch `nn.LSTM` on a synthetic signal.


In [ ]:

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

device = torch.device('cpu')
np.random.seed(0)
torch.manual_seed(0)

# Synthetic signal: trend + seasonality + noise
T = 500
t = np.arange(T)
series = 0.4*(t/T) + np.sin(2*np.pi*t/50) + 0.6*np.sin(2*np.pi*t/12) + np.random.normal(0, 0.08, T)


In [ ]:

# Windowing: create (seq -> next value) samples
def create_dataset(series, window=24):
    X, y = [], []
    for i in range(len(series) - window):
        X.append(series[i:i+window])
        y.append(series[i+window])
    X = np.array(X)
    y = np.array(y)
    X = torch.tensor(X, dtype=torch.float32).unsqueeze(-1)  # (N, window, 1)
    y = torch.tensor(y, dtype=torch.float32).unsqueeze(-1)  # (N, 1)
    return X, y

WINDOW = 24
X, y = create_dataset(series, WINDOW)
N = X.shape[0]
train_N = int(N*0.8)
X_train, y_train = X[:train_N], y[:train_N]
X_test, y_test = X[train_N:], y[train_N:]
print(f"Train: {X_train.shape}, Test: {X_test.shape}")


In [ ]:

# LSTM regressor
class LSTMRegressor(nn.Module):
    def __init__(self, input_size=1, hidden_size=32, num_layers=1):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, 1)
    def forward(self, x):
        out, _ = self.lstm(x)
        out = out[:, -1, :]
        return self.fc(out)

model = LSTMRegressor().to(device)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)


In [ ]:

# Train
EPOCHS = 60
batch_size = 64

for epoch in range(EPOCHS):
    model.train()
    perm = torch.randperm(X_train.size(0))
    epoch_loss = 0.0
    for i in range(0, X_train.size(0), batch_size):
        idx = perm[i:i+batch_size]
        xb = X_train[idx].to(device)
        yb = y_train[idx].to(device)
        optimizer.zero_grad()
        pred = model(xb)
        loss = criterion(pred, yb)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item() * xb.size(0)
    if (epoch+1) % 10 == 0:
        print(f"Epoch {epoch+1}/{EPOCHS} - Train MSE: {epoch_loss/X_train.size(0):.4f}")


In [ ]:

# Evaluate
model.eval()
with torch.no_grad():
    y_pred = model(X_test.to(device)).cpu().squeeze().numpy()
    y_true = y_test.squeeze().numpy()

plt.figure(figsize=(12,4))
plt.plot(y_true, label='True')
plt.plot(y_pred, label='Predicted')
plt.title('LSTM Forecast: Next-step Prediction')
plt.legend()
plt.tight_layout()
plt.show()



Notes:
- Inputs shaped as (batch, seq_len, feature) with `batch_first=True`.
- Sequence-to-one setup: predict next value from prior window.
- Extend with GRU, multi-step forecasting, or exogenous regressors as needed.
